# コホート分析・リテンション分析

顧客の初回購買月ごとにグループ化し、月次の継続率（リテンション）を追跡します。

---

**実行環境:** MySQL 8.0 / Python 3 / pandas  
**DB:** `sql_portfolio`（`SETUP.md` の手順で事前に構築）

## セットアップ

In [1]:
import mysql.connector
import pandas as pd
from IPython.display import display, HTML

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

con = mysql.connector.connect(
    host='localhost', user='root', password='',
    database='sql_portfolio',
    charset='utf8mb4'
)

def run(sql):
    return pd.read_sql(sql, con)


---

## 分析クエリ

### 01. コホート定義（初回購買月の特定）

**ビジネス課題:** 各顧客の初回購買月を特定し、コホートのグループ分けを行う。

**使用テクニック:** `MIN()`, `DATE_FORMAT`

In [2]:
sql = '''
SELECT
    o.customer_id,
    c.customer_name,
    DATE_FORMAT(MIN(o.order_date), '%Y-%m') AS cohort_month,
    MIN(o.order_date)                       AS first_order_date
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.status IN ('paid', 'shipped')
GROUP BY o.customer_id, c.customer_name
ORDER BY cohort_month, o.customer_id;
'''
df = run(sql)
display(df)

C:\Users\willi\AppData\Local\Temp\ipykernel_13624\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,customer_id,customer_name,cohort_month,first_order_date
0,8,中村和也,2024-01,2024-01-22 13:53:00
1,12,山本真由,2024-01,2024-01-12 11:42:00
2,13,松本健二,2024-01,2024-01-09 15:40:00
3,25,高橋翔太,2024-01,2024-01-10 09:54:00
4,27,伊藤遥,2024-01,2024-01-02 20:20:00
5,3,佐藤美咲,2024-02,2024-02-25 08:55:00
6,18,山口愛,2024-02,2024-02-09 19:37:00
7,21,田中彩,2024-02,2024-02-06 20:43:00
8,4,山田大輔,2024-03,2024-03-10 14:10:00
9,24,山田舞,2024-03,2024-03-24 12:32:00


### 02. 購買月とコホート月のマッピング

**ビジネス課題:** 各注文に「初回購買からN ヶ月後」を付与する。

**使用テクニック:** `TIMESTAMPDIFF(MONTH)`

In [3]:
sql = '''
WITH first_purchase AS (
    SELECT
        customer_id,
        DATE_FORMAT(MIN(order_date), '%Y-%m') AS cohort_month,
        MIN(order_date)                       AS first_order_date
    FROM orders
    WHERE status IN ('paid', 'shipped')
    GROUP BY customer_id
)
SELECT
    fp.cohort_month,
    DATE_FORMAT(o.order_date, '%Y-%m')                     AS order_month,
    TIMESTAMPDIFF(MONTH, fp.first_order_date, o.order_date) AS months_since_first,
    o.customer_id,
    o.order_id
FROM orders o
JOIN first_purchase fp ON o.customer_id = fp.customer_id
WHERE o.status IN ('paid', 'shipped')
ORDER BY fp.cohort_month, months_since_first, o.customer_id;
'''
df = run(sql)
display(df)

C:\Users\willi\AppData\Local\Temp\ipykernel_13624\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,cohort_month,order_month,months_since_first,customer_id,order_id
0,2024-01,2024-01,0,8,7
1,2024-01,2024-01,0,12,4
2,2024-01,2024-01,0,13,3
3,2024-01,2024-01,0,25,2
4,2024-01,2024-01,0,27,8
...,...,...,...,...,...
63,2024-10,2024-12,1,1,77
64,2024-11,2024-11,0,23,70
65,2024-12,2024-12,0,10,81
66,2024-12,2024-12,0,28,80


### 03. コホート別リテンションテーブル

**ビジネス課題:** コホート×経過月の2次元マトリクスで残存人数を集計する。

**使用テクニック:** 条件付き集計 `CASE + COUNT DISTINCT`

In [4]:
sql = '''
WITH first_purchase AS (
    SELECT
        customer_id,
        DATE_FORMAT(MIN(order_date), '%Y-%m') AS cohort_month,
        MIN(order_date)                       AS first_order_date
    FROM orders
    WHERE status IN ('paid', 'shipped')
    GROUP BY customer_id
),
order_cohort AS (
    SELECT
        fp.cohort_month,
        TIMESTAMPDIFF(MONTH, fp.first_order_date, o.order_date) AS month_offset,
        o.customer_id
    FROM orders o
    JOIN first_purchase fp ON o.customer_id = fp.customer_id
    WHERE o.status IN ('paid', 'shipped')
)
SELECT
    cohort_month,
    COUNT(DISTINCT CASE WHEN month_offset = 0  THEN customer_id END) AS m0,
    COUNT(DISTINCT CASE WHEN month_offset = 1  THEN customer_id END) AS m1,
    COUNT(DISTINCT CASE WHEN month_offset = 2  THEN customer_id END) AS m2,
    COUNT(DISTINCT CASE WHEN month_offset = 3  THEN customer_id END) AS m3,
    COUNT(DISTINCT CASE WHEN month_offset = 4  THEN customer_id END) AS m4,
    COUNT(DISTINCT CASE WHEN month_offset = 5  THEN customer_id END) AS m5,
    COUNT(DISTINCT CASE WHEN month_offset = 6  THEN customer_id END) AS m6,
    COUNT(DISTINCT CASE WHEN month_offset >= 7 THEN customer_id END) AS m7plus
FROM order_cohort
GROUP BY cohort_month
ORDER BY cohort_month;
'''
df = run(sql)
display(df)

C:\Users\willi\AppData\Local\Temp\ipykernel_13624\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,cohort_month,m0,m1,m2,m3,m4,m5,m6,m7plus
0,2024-01,5,1,2,1,2,0,0,1
1,2024-02,3,0,0,1,0,1,0,1
2,2024-03,2,1,1,0,1,0,0,0
3,2024-04,2,0,1,1,1,0,1,0
4,2024-05,2,1,1,1,2,1,0,0
5,2024-06,2,0,0,1,1,1,0,0
6,2024-07,4,1,0,1,2,1,0,0
7,2024-08,1,0,0,1,0,0,0,0
8,2024-09,1,0,0,0,0,0,0,0
9,2024-10,1,1,0,0,0,0,0,0


### 04. コホート別リテンション率（%）

**ビジネス課題:** 初月人数を分母にした各月残存率を算出する。

**使用テクニック:** CTE 3段, `NULLIF`, `ROUND`

In [5]:
sql = '''
WITH first_purchase AS (
    SELECT
        customer_id,
        DATE_FORMAT(MIN(order_date), '%Y-%m') AS cohort_month,
        MIN(order_date)                       AS first_order_date
    FROM orders
    WHERE status IN ('paid', 'shipped')
    GROUP BY customer_id
),
order_cohort AS (
    SELECT
        fp.cohort_month,
        TIMESTAMPDIFF(MONTH, fp.first_order_date, o.order_date) AS month_offset,
        o.customer_id
    FROM orders o
    JOIN first_purchase fp ON o.customer_id = fp.customer_id
    WHERE o.status IN ('paid', 'shipped')
),
cohort_counts AS (
    SELECT
        cohort_month,
        COUNT(DISTINCT CASE WHEN month_offset = 0 THEN customer_id END) AS m0,
        COUNT(DISTINCT CASE WHEN month_offset = 1 THEN customer_id END) AS m1,
        COUNT(DISTINCT CASE WHEN month_offset = 2 THEN customer_id END) AS m2,
        COUNT(DISTINCT CASE WHEN month_offset = 3 THEN customer_id END) AS m3,
        COUNT(DISTINCT CASE WHEN month_offset = 4 THEN customer_id END) AS m4,
        COUNT(DISTINCT CASE WHEN month_offset = 5 THEN customer_id END) AS m5,
        COUNT(DISTINCT CASE WHEN month_offset = 6 THEN customer_id END) AS m6
    FROM order_cohort
    GROUP BY cohort_month
)
SELECT
    cohort_month,
    m0                                                        AS cohort_size,
    ROUND(m1 / NULLIF(m0, 0) * 100, 1)                       AS m1_pct,
    ROUND(m2 / NULLIF(m0, 0) * 100, 1)                       AS m2_pct,
    ROUND(m3 / NULLIF(m0, 0) * 100, 1)                       AS m3_pct,
    ROUND(m4 / NULLIF(m0, 0) * 100, 1)                       AS m4_pct,
    ROUND(m5 / NULLIF(m0, 0) * 100, 1)                       AS m5_pct,
    ROUND(m6 / NULLIF(m0, 0) * 100, 1)                       AS m6_pct
FROM cohort_counts
ORDER BY cohort_month;
'''
df = run(sql)
display(df)

C:\Users\willi\AppData\Local\Temp\ipykernel_13624\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,cohort_month,cohort_size,m1_pct,m2_pct,m3_pct,m4_pct,m5_pct,m6_pct
0,2024-01,5,20.00,40.00,20.00,40.00,0.00,0.00
1,2024-02,3,0.00,0.00,33.30,0.00,33.30,0.00
2,2024-03,2,50.00,50.00,0.00,50.00,0.00,0.00
3,2024-04,2,0.00,50.00,50.00,50.00,0.00,50.00
4,2024-05,2,50.00,50.00,50.00,100.00,50.00,0.00
5,2024-06,2,0.00,0.00,50.00,50.00,50.00,0.00
6,2024-07,4,25.00,0.00,25.00,50.00,25.00,0.00
7,2024-08,1,0.00,0.00,100.00,0.00,0.00,0.00
8,2024-09,1,0.00,0.00,0.00,0.00,0.00,0.00
9,2024-10,1,100.00,0.00,0.00,0.00,0.00,0.00


### 05. 経過月別 平均リテンション率サマリ

**ビジネス課題:** 全コホートを横断した平均残存率で傾向を把握する。

**使用テクニック:** `AVG()`, サブクエリ

In [6]:
sql = '''
WITH first_purchase AS (
    SELECT
        customer_id,
        MIN(order_date) AS first_order_date
    FROM orders
    WHERE status IN ('paid', 'shipped')
    GROUP BY customer_id
),
order_cohort AS (
    SELECT
        fp.customer_id,
        TIMESTAMPDIFF(MONTH, fp.first_order_date, o.order_date) AS month_offset
    FROM orders o
    JOIN first_purchase fp ON o.customer_id = fp.customer_id
    WHERE o.status IN ('paid', 'shipped')
),
cohort_stats AS (
    -- 各顧客が month_offset = N に購買したかどうか（0/1）
    SELECT
        oc.customer_id,
        oc.month_offset,
        1 AS active
    FROM order_cohort oc
    WHERE oc.month_offset > 0
    GROUP BY oc.customer_id, oc.month_offset
)
SELECT
    cs.month_offset                                AS months_after_first,
    COUNT(DISTINCT cs.customer_id)                 AS active_customers,
    (SELECT COUNT(DISTINCT customer_id)
     FROM first_purchase)                          AS total_customers,
    ROUND(
        COUNT(DISTINCT cs.customer_id) /
        NULLIF((SELECT COUNT(DISTINCT customer_id) FROM first_purchase), 0)
        * 100, 1
    )                                              AS avg_retention_pct
FROM cohort_stats cs
GROUP BY cs.month_offset
ORDER BY cs.month_offset;
'''
df = run(sql)
display(df)

C:\Users\willi\AppData\Local\Temp\ipykernel_13624\2420247800.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, con)


,months_after_first,active_customers,total_customers,avg_retention_pct
0,1,5,26,19.20
1,2,5,26,19.20
2,3,7,26,26.90
3,4,9,26,34.60
4,5,4,26,15.40
5,6,1,26,3.80
6,9,2,26,7.70
7,10,1,26,3.80


---

In [7]:
con.close()
print('接続を閉じました。')

接続を閉じました。
